# Introduction
This notebook is a clone of the run_quantize.py file for mimicking and debugging experiments without using seml and slurm

## 1. Initial Setup

In [1]:
print("Hello World")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Hello World
Free GPU Memory (GB): 10.9082


In [2]:
print("\n################################")
print("Setting up environment...")
print("################################\n")

import os
#os.chdir('..')
print("Current Working Directory ", os.getcwd())
import sys
sys.path.append("../") # Add directory containing src/data to path

import importlib
import src  # Assuming src is the package name

# Reload the src module after making changes
importlib.reload(src)

%load_ext autoreload
%autoreload 2

import seml
import re
import shutil

os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Disables parallelism to remove transformers warning

print("\n################################")
print("Setting up cache paths...")
print("################################\n")

os.environ["MKL_SERVICE_FORCE_INTEL"] = "1"
CACHE_PATH = "/nfs/students/daro/.cache/huggingface"
print(f"Setting cache path to {CACHE_PATH}")

os.environ["TORCH_HOME"] = CACHE_PATH
os.environ["HF_HOME"] = CACHE_PATH

import torch
torch.hub.set_dir(CACHE_PATH)
with torch.no_grad():
    torch.cuda.empty_cache()
    
import logging
logger = logging.getLogger("quant_logger")
    
!cat /proc/meminfo | awk '/MemTotal/ {total=$2} /MemFree/ {free=$2} /MemAvailable/ {available=$2} END {printf "MemTotal: %.2f GB\nMemFree: %.2f GB\nMemAvailable: %.2f GB\n", total/1024/1024, free/1024/1024, available/1024/1024}'
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

# Code formatting and linting

# !black notebooks/Llama-3-8B-quant.ipynb
# !pylint notebooks/Llama-3-8B-quant.ipynb

print("\n################################")
print("Setting up cuda devices...")
print("################################\n")

if torch.cuda.is_available():
    print("CUDA device is available!")
    # Get the number of available CUDA devices
    num_cuda_devices = torch.cuda.device_count()
    print(f"Number of CUDA devices: {num_cuda_devices}")
    
    # Loop through available devices and get name
    for device_id in range(num_cuda_devices):
        device = torch.device(f"cuda:{device_id}")
        name = torch.cuda.get_device_name(device)
        print(f"  - CUDA Device {device_id+1}: {name}")
else:
    print("CUDA device is not available.")
    
print("\n################################")
print("Authentication with Hugging Face...")
print("################################\n")

import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
huggingface_token = os.getenv('HUGGINGFACE_TOKEN')

if huggingface_token is None:
    raise ValueError("Please set the HUGGINGFACE_TOKEN environment variable.")
else:
    print("Hugging Face token loaded successfully.")

login(token=huggingface_token, add_to_git_credential=True)
print("Successfully authenticated with the Hugging Face API.")

print("\n################################")
print("Setting up GPU memory usage list...")
print("################################\n")
# Global list to store GPU memory usage
from src.evaluations.evaluate_memory import record_gpu_memory
gpu_memory_usage = {}
record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Warm up notebook")


################################
Setting up environment...
################################

Current Working Directory  /nfs/homedirs/daro/git/quantization-reliability

################################
Setting up cache paths...
################################

Setting cache path to /nfs/students/daro/.cache/huggingface
MemTotal: 754.56 GB
MemFree: 18.75 GB
MemAvailable: 740.29 GB
Free GPU Memory (GB): 10.9082

################################
Setting up cuda devices...
################################

CUDA device is available!
Number of CUDA devices: 1
  - CUDA Device 1: NVIDIA GeForce GTX 1080 Ti

################################
Authentication with Hugging Face...
################################



/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Hugging Face token loaded successfully.
Token is valid (permission: write).
Your token has been saved in your configured git credential helpers (store).
Your token has been saved to /nfs/students/daro/.cache/huggingface/token
Login successful
Successfully authenticated with the Hugging Face API.

################################
Setting up GPU memory usage list...
################################

Free GPU Memory (GB): 10.9062. Context: Warm up notebook.


## 3. SEML Pipeline

In [3]:
from src.models import get_model, get_model_name, get_tokenizer
from src.data import get_dataset, data_loader_from_split
from src.algorithms.quantization.quantize import quantize
from src.evaluations.evaluate_all import evaluate

from src.evaluations.evaluate_memory import record_gpu_memory
gpu_memory_usage = {}

def run_quantize(
    # Dataset parameters
    seed_dataset=123,
    directory_dataset="",
    calib_dataset_name="",
    calib_dataset_split="",
    eval_dataset_name="",
    eval_dataset_split="",
    batch_size=1,
    dataset_stride=1024,
    dataset_seq_length=1024,
    # Model parameters
    seed_model=123,
    directory_model="",
    clean_cache=True,
    model_name="",
    # Quantization parameters
    quantize_method="",
    num_bits=8,
    # Evaluation metrics
    eval_metrics=[
        "perplexity",
        "brier_score",
        # "model_size",
        # "gpu_utilization"
    ],
    device="cuda",
    save_quantized_model=False,
    quantized_model_save_path="",
):
    ##################
    ## Print config ##
    ##################
    logger.info("Received the following configuration:")
    logger.info(
        f"Calibration dataset: {calib_dataset_name}\n"
        f"Calibration split: {calib_dataset_split}\n"
        f"Evaluation dataset: {eval_dataset_name}\n"
        f"Evaluation split: {eval_dataset_split}\n"
        f"Dataloader stride: {dataset_stride}\n"
        f"Dataloader sequence length: {dataset_seq_length}\n"
        f"Batch size: {batch_size}\n"
        f"Model: {model_name}\n"
        f"Quantize method: {quantize_method}\n"
        f"Quantize bits: {num_bits}\n"
        f"Evaluation metrics: {eval_metrics}\n"
        f"Device: {device}\n"
        f"Quantized model save: {save_quantized_model}\n"
        f"Quantized model save path: {quantized_model_save_path}\n"
    )
    
    ####################
    ## Load tokenizer ##
    ####################
    logger.info("Load tokenizer")
    model_full_name = get_model_name(model_name)
    tokenizer = get_tokenizer(
        model_name=model_full_name,
        seed=seed_model,
        directory_model=directory_model,
        device=device,
    )
    record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Load tokenizer")
    
    ###############
    ## Load data ##
    ###############
    logger.info("Load calibration data module")
    calib_data_module = get_dataset(
        dataset_name=calib_dataset_name,
        directory_dataset=directory_dataset,
        batch_size=batch_size,
        sequence_length=dataset_stride,
        tokenizer_name=model_full_name,
        seed=seed_dataset,
    )
    logger.info("Load evaluation data module")
    eval_data_module = get_dataset(
        dataset_name=eval_dataset_name,
        directory_dataset=directory_dataset,
        batch_size=batch_size,
        sequence_length=dataset_stride,
        tokenizer_name=model_full_name,
        seed=seed_dataset,
    )
    
    logger.info("Load calibration dataloader")
    calib_dataloader = data_loader_from_split(calib_data_module)[calib_dataset_split]
    logger.info("Load evaluation dataloader")
    eval_dataloader = data_loader_from_split(eval_data_module)[eval_dataset_split]
    record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Load data")

    ##############
    ## Quantize ##
    ##############
    logger.info("Quantization...")
    quantized_model = None
    if quantize_method == "NONE":
        logger.info("Quantize method is None, loading original model")
        quantized_model = get_model(
            model_name=model_full_name,
            seed=seed_model,
            directory_model=directory_model,
            device=device,
        )
        record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Load base model")
    else:
        logger.info(f"Quantizing model using {quantize_method}")
        quantized_model = quantize(
            model_name=model_full_name,
            tokenizer=tokenizer,
            calib_dataloader=calib_dataloader,
            quantize_method=quantize_method,
            num_bits=num_bits,
            save_model=save_quantized_model,
            save_path=quantized_model_save_path,
            device=device
        )
        record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Quantize model")

    ##############
    ## Evaluate ##
    ##############
    logger.info("Evaluation...")
    results = evaluate(
        model=quantized_model,
        eval_dataloader=eval_dataloader,
        eval_metrics=eval_metrics,
        factor=1,
        device=device,
        to_device=(quantize_method in ["AWQ"]),
        prefix="",
        gpu_memory_usage=gpu_memory_usage
    )
    record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Evaluate model")
    
    ####################
    ## Cleaning cache ##
    ####################
    logger.info("Cleaning cache...")
    if clean_cache:
        for root, dirs, files in os.walk(CACHE_PATH, topdown=False):
            for dir_name in dirs:
                pattern = re.compile(f"^.*{model_full_name.split('/')[-1]}.*")
                dir_path = os.path.join(root, dir_name)
                if re.match(pattern, dir_path):
                    try:
                        shutil.rmtree(dir_path)
                    except:
                        pass

    fail_trace = {
        "fail_trace": seml.evaluation.get_results,
    }

    return {**results, **fail_trace}

In [5]:
import itertools
import random
import torch  # Ensure torch is imported

# Fixed parameters
fixed_params = {
    'device': 'cuda',
    'clean_cache': True,
    'save_quantized_model': True,
    'seed_model': 123,
    'seed_dataset': 123,
    'batch_size': 1,
    'dataset_stride': 1024,
    'dataset_seq_length': 1024,
    'eval_metrics': ['perplexity', 'brier_score', 'quantize_runtime', 'model_size'],
    'calib_dataset_split': 'validation',
    'eval_dataset_split': 'test',
}

# Grid parameters
grid_params = {
    'calib_dataset_name': ['WikiText', 'OpenAssistant'],
    'eval_dataset_name': ['WikiText', 'OpenAssistant'],
    'quantize_method': ['NONE', 'HQQ'],
    'num_bits': [8],
    'model_name': ['TinyLlama']
}

batch_sizes = [1]

# Generate all combinations for grid search
grid_combinations = list(itertools.product(
    grid_params['calib_dataset_name'],
    grid_params['eval_dataset_name'],
    grid_params['quantize_method'],
    grid_params['num_bits'],
    grid_params['model_name']
))

# Run the quantize function for all combinations
results = []
max_combinations = 10000  # Set to a lower number for testing purposes
for i, combination in enumerate(grid_combinations):
    if i >= max_combinations:
        break
    for batch_size in batch_sizes:
        calib_dataset_name, eval_dataset_name, quantize_method, num_bits, model_name = combination

        # Print current combination details
        print(f"Running combination {i+1}/{len(grid_combinations)}")
        print(f"  Model Name: {model_name}")
        print(f"  Calibration Dataset: {calib_dataset_name}")
        print(f"  Evaluation Dataset: {eval_dataset_name}")
        print(f"  Quantize Method: {quantize_method}")
        print(f"  Num Bits: {num_bits}")
        print(f"  Batch Size: {batch_size}")

        result = run_quantize(
            # Fixed parameters
            device=fixed_params['device'],
            clean_cache=fixed_params['clean_cache'],
            save_quantized_model=fixed_params['save_quantized_model'],
            seed_model=fixed_params['seed_model'],
            seed_dataset=fixed_params['seed_dataset'],
            eval_metrics=fixed_params['eval_metrics'],
            calib_dataset_split=fixed_params['calib_dataset_split'],
            eval_dataset_split=fixed_params['eval_dataset_split'],
            # Grid parameters
            calib_dataset_name=calib_dataset_name,
            eval_dataset_name=eval_dataset_name,
            quantize_method=quantize_method,
            num_bits=num_bits,
            model_name=model_name,
            # Random parameters
            batch_size=batch_size,
            dataset_stride=fixed_params['dataset_stride'],
            dataset_seq_length=fixed_params['dataset_seq_length'],
            # Model parameters
            directory_model="",
            directory_dataset="",
            quantized_model_save_path=""
        )

        # Append result with parameter details
        results.append({
            'result': result,
            'parameters': {
                'model_name': model_name,
                'calib_dataset_name': calib_dataset_name,
                'eval_dataset_name': eval_dataset_name,
                'quantize_method': quantize_method,
                'num_bits': num_bits,
                'batch_size': batch_size,
                'dataset_stride': fixed_params['dataset_stride'],
                'dataset_seq_length': fixed_params['dataset_seq_length'],
            }
        })

# Do something with the results
print(results)

Running combination 1/8
  Model Name: TinyLlama
  Calibration Dataset: WikiText
  Evaluation Dataset: WikiText
  Quantize Method: NONE
  Num Bits: 8
  Batch Size: 1
Free GPU Memory (GB): 10.9062. Context: Load tokenizer.


Token indices sequence length is longer than the specified maximum sequence length for this model (2874559 > 2048). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (2874559 > 2048). Running this sequence through the model will result in indexing errors


Free GPU Memory (GB): 10.9062. Context: Load data.
Free GPU Memory (GB): 6.3633. Context: Load base model.
Free GPU Memory (GB): 6.3633. Context: Evaluate GPU type.
Model in evaluation mode. Device: cuda
Processing batch 0
Processing batch 1
Processing batch 2
Processing batch 3
Processing batch 4
Processing batch 5
Processing batch 6
Processing batch 7
Processing batch 8
Processing batch 9
Processing batch 10
Processing batch 11
Processing batch 12
Processing batch 13
Processing batch 14
Processing batch 15
Processing batch 16
Processing batch 17
Processing batch 18
Processing batch 19
Processing batch 20
Processing batch 21
Processing batch 22
Processing batch 23
Processing batch 24
Processing batch 25
Processing batch 26
Processing batch 27
Processing batch 28
Processing batch 29
Processing batch 30
Processing batch 31
Processing batch 32
Processing batch 33
Processing batch 34
Processing batch 35
Processing batch 36
Processing batch 37
Processing batch 38
Processing batch 39
Proces

NameError: name 're' is not defined

In [7]:
max_combinations

1

In [8]:
i

0

In [9]:
batch_sizes

[1]